In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/employee'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Extracting the Data for initial Exploration

### import the necesary header files 

In [ ]:
#for data manipulation
import pandas as pd 
import numpy as np
from collections import Counter

#for better analysis of 
import seaborn as sns 
sns.set()
import plotly.express as px
import matplotlib.pyplot as plt 
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#for model definition and testing
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

### Read the dataset to a variable df => DataFrame

In [ ]:
#pandas.read_csv => used to read a csv data file
df = pd.read_csv('/kaggle/input/employees-satisfaction-analysis/Employee Attrition.csv')
#show the first 10 rows of the imported data 
df.head(10)

### df.info() shows the basic information related to the entire dataset. These include number of non-null values, Column names and data type of each column

In [ ]:
df.info()
#non required columns - Emp ID

#### **df.describe()** The function is used to describe the statistics of the dataset. It mainly shows the total count, mean, median and mode of each columns in the dataset.

In [ ]:
df.describe().T

Here we can see that there are 14,999 employees surveyed as part of this analysis. First let's explore all the columns:

**Emp Id** It is unique for all of them starting from 1 to 14,999. So it is clear that the employee numer does not depend on satisfaction level of an employee.

**Satisfaction Level:** The average satisfaction level of the employees is 0.613 or it can be interpreted as the average satisfaction level of an employee is about 61% with a standard deviation of 0.249. 25% of employees have a satisfaction level below 0.44 and 75% of employees have a satisfaction level below 0.82.

**Number of projects:** On an average an employee gets about 3.8 projects to work on with a std deviation of 1.2. 25% of employees have less than 3 projects and 75% of employees have less than 5 projects. The highest number of projects for an employee is 7 whereas the minimum number of project is 2. The number of project might have a strong dependence on satisfaction level since this reveals the workload distribution among employees.

**Last Evaluation:** It can be interpreted as the performance score of the employees. This statistics helps in determining the productiveness of the employees. On an average the performance score of employees is 0.71. The quartile measures reveals, only 25% of employees have a performance score above 0.82. Half of the employees have a performance score above 0.72 and half of them have performance score below 0.7. The least productive 25% have a performance score of 0.56.

**Time spend in company:** On an average an employee stay for upto 3 years in a company. The maximum an employee spent in a company is for 10 years. More than 75% of employees stay in a company for less than 4 years. At a minimum they stay for 2 years.

Similarly we could see the statistics of other columns also

# Data Cleaning

#### Since the employee id is continuos from 1 to 14,999 and does not add any information to analysis we could make it as the index of the dataset.

In [ ]:
newDf = df.set_index('Emp ID')
newDf.info()

In [ ]:
#replace the newDf with df
df = newDf
df.info()

In [ ]:
df.describe().T

In [ ]:
#converting the dtype of salary and and dept to category variable
cols_to_be_categorical = ['dept','salary']
for value in cols_to_be_categorical:
    df[value]=df[value].astype('category')

In [ ]:
df.info()

### Dealing with missing values and handling it 

In [ ]:
#finding the missing values 
missing_values = df[df.isna().any(axis=1)]
display(missing_values.head())
display(missing_values.isna().count())

The 788 entries have no data. So we could simply drop those rows. It is done as a part of handling missing values or null values. If null values are not properly handled it can cause errors or false results in model development and testing

In [ ]:
#there are 788 entries with missing values. Since all those 788 rows does not have any values we can simply drop thdf.dropna(inplace = True)
df.dropna(inplace=True)

In [ ]:
df.info()

# Detecting outliers in the data

**An outlier is an observation that lies an abnormal distance from other values in a random sample from a population.**

**A box plot is constructed by drawing a box between the upper and lower quartiles with a solid line drawn across the box to locate the median.**

**Let there be Q1, Q2 and Q3 be quartile**

***
**IQR = > InterQuartile Range = Q3 - Q1**

**upperFence = Q3 + 1.5xIQR**

**LowerFence =  Q1 - 1.5xIQR**
***

**Anything that falls above the upperFence and anything that falls below the lowerfence are considered as outliers. If necessary we could remove outliers. But in most cases we keep outliers since there are these kind of variation in all real-world data**

In [ ]:
#number of outliers in the data
def outliers(data):
    Q1,Q2,Q3 = np.quantile(data, [.25,.5,.75])
    IQR = Q3 - Q1
    UpperFence = Q3 + (1.5*IQR)
    LowerFence = Q1 - (1.5*IQR)
    return data[ (data > UpperFence) | (data < LowerFence)]


_columns = df.select_dtypes(np.number).columns
for _cols in _columns:
    result = outliers(df[_cols].values)
    print(_cols,":",Counter(result))


In [ ]:
#detecting outliers using Box Plot 
fig = make_subplots(
    rows = 1, cols = len(_columns),    
)

for i, _col in enumerate(_columns,start=1):
    y0 = df[_col].values
    fig.append_trace(go.Box(y=y0,name=_col), row = 1,col = i)

fig.update_layout(height=500, width=1200, title_text="Outliers")
fig.show()

## EDA - Exploratory Data Analysis

In [ ]:
px.histogram(df['number_project'],nbins = len(df['number_project'].unique()),
            x = 'number_project',
            title = 'Employee_count v/s number of projects',
            )

In [ ]:
#Analysis on Categorical Data
categoricalColumns = df.select_dtypes('category').columns

df[categoricalColumns[0]].value_counts()
fig = px.bar(df[categoricalColumns[0]].value_counts(),
             color=df[categoricalColumns[0]].unique(),
             title = "Employees in each department",
            )
fig.show()

fig = px.bar(df[categoricalColumns[1]].value_counts(),
            color = df[categoricalColumns[1]].unique(),
            title = "Employees with salary range")
fig.show()

In [ ]:
styles = [dict(selector="caption",
            props=[("text-align", "left"),
                   ("font-size", "150%"),
                   ("color", 'red')])]


newDf = df.drop('dept',axis = 1).copy()
display(newDf.groupby("salary").mean().style.format("{:.3f}").\
    background_gradient(cmap="Reds",axis=0).\
       set_caption("Satisfaction level with the range of salaies").\
        set_table_styles(styles))
print("\n\n")


#since our dependent variable is satisfaction_rate let's
#analysis only the dependency of that variable alone
df.groupby(["salary","dept"])[["satisfaction_level"]].mean().unstack().\
    style.format("{:.3f}").background_gradient(cmap="Blues",axis=0).\
    set_caption("Salary v/s satisfaction rate of each dept").set_table_styles(styles)


In [ ]:


#Variation of salary in each dept for each dependencies other than satisfaction_level
for _col in _columns:
    if _col == 'satisfaction_level':
        continue
    display(df.groupby(["salary","dept"])[[_col]].mean().unstack().\
           style.format("{:.3f}").background_gradient(cmap="Blues").\
           set_caption(f"Salary v/s {_col}").set_table_styles(styles))
    print("\n")

## Model Development

### Defining Correlation matrix to find the dependencies of each features on one another

A correlation matrix is a statistical technique used to evaluate the relationship between two variables in a data set. The matrix is a table in which every cell contains a correlation coefficient, where 1 is considered a strong relationship between variables, 0 a neutral relationship and -1 a not strong relationship.

In [ ]:
df2 = df[['satisfaction_level','last_evaluation','number_project','average_montly_hours','time_spend_company','Work_accident','promotion_last_5years']].corr()

In [ ]:
plt.figure(figsize = (30,15))
sns.heatmap(df2.corr(method="pearson"),annot = True)
sns.set(font_scale = 1.5)

### Label encoding for categorical data for Model Development

In [ ]:
#label encoding for categorical data
labelEncoder = LabelEncoder()
for value in categoricalColumns:
    df[value] = labelEncoder.fit_transform(df[value])

#### Define the dependent and independent variables. The aim of this analysis is to predict the satisfaction level of an employee. So the dependent variable or target vector X is satisfaction level. Satisfaction level depends on every other columns to a certain extent. So rest of the columns are treated as feature vectors.

**General Regression analysis equation is given as**
### $Y = β_{0} + βX$
 $y = > Target Vector$
 
 $x =>  Feature$
 
 Here since the X is array of features we use
 ### $Y=β^{T}_{i}.X_{i}+ β_{0}$
 
  $X_{i} => feature Vectors$
  $β_{0} => Bias$
  $β{T}_{i} => Weights$
  $Y => Target Vector$

In [ ]:
#setting up of target variable and feature variable
newDf = df.copy() 

#let target = y,  and features = X
y = newDf['satisfaction_level']
display(y)

newDf.drop('satisfaction_level',axis=1,inplace=True)
X = newDf
display(X)

In [ ]:
#splitting the dataset to train_test_split
X_train,X_test, y_train,y_test = train_test_split(X,y,shuffle = True)

In [ ]:


tree = DecisionTreeRegressor(max_depth=8,random_state=40)
tree.fit(X_train,y_train)

In [ ]:
print(tree.score(X_train,y_train))
print(tree.score(X_test,y_test))

In [ ]:
# applying cost complexity pruning inorder to reduce the overfitting
path = tree.cost_complexity_pruning_path(X_train,y_train)
alphas = path['ccp_alphas'].round(5)
print(alphas)

In [ ]:
train_score,test_score = [],[]
for alpha in alphas:
    decisionTree = DecisionTreeRegressor(ccp_alpha = alpha,max_depth=8)
    decisionTree.fit(X_train,y_train)
    
    train_score.append(decisionTree.score(X_train,y_train))
    test_score.append(decisionTree.score(X_test,y_test))
    
    

In [ ]:
fig = plt.figure(figsize=(20,10))
sns.lineplot(x=alphas, y=train_score,label="Training Accuracy")
sns.lineplot(x=alphas, y=test_score,label="Testing Accuracy")
plt.vlines(alphas[np.argmax(test_score)],0,0.6)
fig.show()

In [ ]:
#so now fit the decision tree with max_test_score
max_test_score_alpha = alphas[np.argmax(test_score)]
dtree = DecisionTreeRegressor(ccp_alpha = max_test_score_alpha,
                                  max_depth=8,random_state = 40)

dtree.fit(X_train,y_train)


In [ ]:
#updated test_score and training score 

print(dtree.score(X_train,y_train))
print(dtree.score(X_test,y_test))

In [ ]:
#R-squared Value
print(f"The R-squared value : {dtree.score(X_test,y_test):.2%}")